In [ ]:
# =========================================================
# IMPORTS
# =========================================================
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from scipy.stats import combine_pvalues
import requests
import json
import statsmodels.api as sm
import re

# =========================================================
# CONFIG
# =========================================================
DATA_DIR = Path("data/tcga_maf")
OUTPUT_DIR = Path("data/tcga_omics")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 8

DATA_TYPES = {
    "mirna": "miRNA Expression Quantification",
}

# GDC_FILES_API = "https://api.gdc.cancer.gov/files"
# GDC_DATA_API = "https://api.gdc.cancer.gov/data"

GDC_API_FILES = "https://api.gdc.cancer.gov/files"
GDC_API_DATA = "https://api.gdc.cancer.gov/data"

CANCERS = ["TCGA-BRCA", "TCGA-LUAD", "TCGA-LUSC", "TCGA-HNSC"]

from pathlib import Path
import json
import requests
import numpy as np
import pandas as pd
# import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
# from statsmodels.stats.multitest import multipletests

# =========================
# Paths & Constants
# =========================

OUT_DIR = Path("../data/gdc_cache")
DATA_DIR = Path("data/tcga_mirna")
OUTPUT_DIR = Path("data/tcga_mirna")

for d in [OUT_DIR, DATA_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 8

CANCERS = [
    "TCGA-BLCA",  # Bladder Urothelial Carcinoma
    "TCGA-BRCA",  # Breast Invasive Carcinoma
    "TCGA-CESC",  # Cervical Squamous Cell Carcinoma
    "TCGA-CHOL",  # Cholangiocarcinoma
    "TCGA-COAD",  # Colon Adenocarcinoma
    "TCGA-DLBC",  # Diffuse Large B-cell Lymphoma
    "TCGA-ESCA",  # Esophageal Carcinoma
    "TCGA-GBM",   # Glioblastoma Multiforme
    "TCGA-HNSC",  # Head and Neck Squamous Cell Carcinoma
    "TCGA-KICH",  # Kidney Chromophobe
    "TCGA-KIRC",  # Kidney Renal Clear Cell Carcinoma
    "TCGA-KIRP",  # Kidney Renal Papillary Cell Carcinoma
    "TCGA-LAML",  # Acute Myeloid Leukemia
    "TCGA-LGG",   # Lower Grade Glioma
    "TCGA-LIHC",  # Liver Hepatocellular Carcinoma
    "TCGA-LUAD",  # Lung Adenocarcinoma
    "TCGA-LUSC",  # Lung Squamous Cell Carcinoma
    "TCGA-MESO",  # Mesothelioma
    "TCGA-OV",    # Ovarian Serous Cystadenocarcinoma
    "TCGA-PAAD",  # Pancreatic Adenocarcinoma
    "TCGA-PCPG",  # Pheochromocytoma and Paraganglioma
    "TCGA-PRAD",  # Prostate Adenocarcinoma
    "TCGA-READ",  # Rectum Adenocarcinoma
    "TCGA-SARC",  # Sarcoma
    "TCGA-SKCM",  # Skin Cutaneous Melanoma
    "TCGA-STAD",  # Stomach Adenocarcinoma
    "TCGA-TGCT",  # Testicular Germ Cell Tumors
    "TCGA-THCA",  # Thyroid Carcinoma
    "TCGA-THYM",  # Thymoma
    "TCGA-UCEC",  # Uterine Corpus Endometrial Carcinoma
    "TCGA-UCS",   # Uterine Carcinosarcoma
    "TCGA-UVM"    # Uveal Melanoma
]

GDC_API_FILES = "https://api.gdc.cancer.gov/files"
GDC_API_DATA = "https://api.gdc.cancer.gov/data"

# =========================
# miRTarBase Sources
# =========================

MIRTARBASE_FILES = {
    "SE_WR":   ("../data/miRTarBase_SE_WR.csv", 3.0),
    "SE_W":    ("../data/miRTarBase_SE_W.csv", 2.0),
    "SE_R":    ("../data/miRTarBase_SE_R.csv", 2.0),
    "WE_CLIP": ("../data/miRTarBase_WE_Clip.csv", 1.0),
    "WE_OTHER":("../data/miRTarBase_WE_Other.csv", 1.0),
}


def query_files(project, data_type, size=1000):
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
            {"op": "in", "content": {"field": "data_type", "value": [data_type]}},
        ],
    }

    params = {
        "filters": json.dumps(filters),
        "format": "JSON",
        "size": str(size),
        "fields": "file_id,file_name,created_datetime",
        "sort": "created_datetime:desc"   
    }

    r = requests.get(GDC_API_FILES, params=params)
    r.raise_for_status()
    hits = r.json()["data"]["hits"]

    return [(h["file_id"], h["file_name"], h["created_datetime"]) for h in hits]

def load_mirtarbase(mirtarbase_files):
    """
    Load miRTarBase files and return:
        mirna -> list of (gene, weight)
    """
    mapping = {}

    if isinstance(mirtarbase_files, str):
        mirtarbase_files = {"default": (mirtarbase_files, 1.0)}

    for name, (path, weight) in mirtarbase_files.items():
        try:
            df = pd.read_excel(path) if path.endswith(".xlsx") else pd.read_csv(path)

            mir_col = next((c for c in df.columns if "mirna" in c.lower()), None)
            gene_col = next((c for c in df.columns if "target" in c.lower()), None)

            if not mir_col or not gene_col:
                print(f"[WARN] Skipping {name}: missing columns")
                continue

            for _, row in df.iterrows():
                mir = str(row[mir_col]).lower().strip()
                gene = str(row[gene_col]).upper().strip()

                if mir and gene and gene != "NAN":
                    mapping.setdefault(mir, []).append((gene, weight))

            print(f"[INFO] Loaded {name}: {len(df)} rows")

        except Exception as e:
            print(f"[WARN] Failed {name}: {e}")

    print(f"[INFO] Total miRNAs loaded: {len(mapping)}")
    return mapping

# =========================
# Utilities
# =========================

def parse_tcga_barcode(barcode: str) -> str:
    """
    Extract sample type from TCGA barcode.

    Examples:
        TCGA-XX-XXXX-01A → tumor
        TCGA-XX-XXXX-11A → normal
    """
    if not isinstance(barcode, str):
        return "other"

    parts = barcode.split("-")
    if len(parts) < 4:
        return "other"

    sample_code = parts[3][:2]

    if sample_code == "01":
        return "tumor"
    elif sample_code == "11":
        return "normal"
    return "other"

def query_gdc(project, data_type):
    cache_file = OUT_DIR / f"{project}_{data_type.replace(' ', '_')}.json"

    if cache_file.exists():
        print(f"[cache] {data_type}: {project}")
        with open(cache_file) as f:
            return json.load(f)

    print(f"[download] {data_type}: {project}")

    params = {
        "filters": {
            "op": "and",
            "content": [
                {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
                {"op": "in", "content": {"field": "data_type", "value": [data_type]}}
            ]
        },
        "fields": "file_id,file_name",
        "format": "JSON",
        "size": 200
    }

    r = requests.post(GDC_API_FILES, json=params)
    r.raise_for_status()

    hits = r.json()["data"]["hits"]
    files = [(f["file_id"], f["file_name"]) for f in hits]

    with open(cache_file, "w") as f:
        json.dump(files, f)

    return files


def combat_batch_correction(mat: pd.DataFrame) -> pd.DataFrame:
    """
    Lightweight batch correction using per-sample z-score normalization.
    """
    scaler = StandardScaler(with_mean=True, with_std=True)

    return pd.DataFrame(
        scaler.fit_transform(mat.T).T,
        index=mat.index,
        columns=mat.columns
    )


# =========================
# Differential Expression
# =========================

def process_mirna_limma(mat: pd.DataFrame,
                       sample_info: dict,
                       min_var: float = 1e-5) -> pd.DataFrame:
    print(f"[DEBUG] matrix: {mat.shape}")

    if mat.empty:
        return pd.DataFrame()

    # Variance filter
    mat = mat[mat.var(axis=1) > min_var]
    print(f"[DEBUG] after variance filter: {mat.shape}")

    group = []
    keep_cols = []

    for sample in mat.columns:
        sample_type = sample_info.get(sample, {}).get("type", "other")
        if sample_type in ("tumor", "normal"):
            group.append(1 if sample_type == "tumor" else 0)
            keep_cols.append(sample)

    if len(keep_cols) < 6:
        print("[WARN] not enough tumor/normal samples")
        return pd.DataFrame()

    mat = mat[keep_cols]
    group = np.array(group)

    print(f"[INFO] tumor: {(group==1).sum()} | normal: {(group==0).sum()}")

    # Batch correction
    mat = combat_batch_correction(mat)

    # Design matrix
    X = sm.add_constant(group)

    logFC, pvals = [], []

    for gene in mat.index:
        y = mat.loc[gene].values

        try:
            model = sm.OLS(y, X).fit()
            logFC.append(model.params[1])
            pvals.append(model.pvalues[1])
        except Exception:
            logFC.append(0.0)
            pvals.append(1.0)

    logFC = pd.Series(logFC, index=mat.index)
    pval = pd.Series(pvals, index=mat.index)

    # FDR correction
    _, fdr_vals, _, _ = multipletests(pval.values, method="fdr_bh")
    fdr = pd.Series(fdr_vals, index=mat.index)

    score = logFC * (-np.log10(fdr + 1e-10))

    return pd.DataFrame({
        "mirna": mat.index,
        "logFC": logFC,
        "pval": pval,
        "fdr": fdr,
        "score": score
    }).sort_values("fdr")


# =========================
# GDC Metadata
# =========================

def get_sample_info_from_gdc(file_id: str):
    """
    Returns (barcode, type)
    """
    meta = query_gdc(file_id) 

    try:
        sample = meta["cases"][0]["samples"][0]

        barcode = sample["submitter_id"]
        stype = sample["sample_type"]

        if stype == "Primary Tumor":
            return barcode, "tumor"
        elif stype == "Solid Tissue Normal":
            return barcode, "normal"
        return barcode, "other"

    except (KeyError, IndexError, TypeError):
        return None, None


# =========================
# Data Loading
# =========================

def load_mirna_matrix(project: str,
                      file_ids: list,
                      file_map: dict):
    """
    Build miRNA expression matrix for a project.
    """
    cancer_dir = DATA_DIR / project

    mats = []
    sample_info = {}

    for fid, fname in file_ids:
        fpath = cancer_dir / fname

        if not fpath.exists():
            continue

        try:
            # df = pd.read_csv(fpath, sep="\t", low_memory=False)
            try:
                df = pd.read_csv(fpath, sep="\t", low_memory=False)
                if df.shape[1] == 1:
                    raise ValueError("Wrong delimiter")
            except:
                df = pd.read_csv(fpath, low_memory=False)

            # mirna_col = next((c for c in df.columns if "mirna" in c.lower()), None)
            # expr_col = next((c for c in df.columns if "read" in c.lower() or "rpm" in c.lower()), None)

            mirna_col = next((c for c in df.columns if "mir" in c.lower()), None)
            expr_col = next((c for c in df.columns if any(x in c.lower() for x in ["read", "rpm", "count"])), None)

            if not mirna_col or not expr_col:
                continue

            df = df[[mirna_col, expr_col]].dropna()

            barcode, sample_type = get_sample_info_from_gdc(fid)
            if barcode is None:
                barcode = f"unknown_{fid}"
                sample_type = "other"

            df.columns = ["miRNA", barcode]
            df = df.set_index("miRNA")

            mats.append(df)
            sample_info[barcode] = {"type": sample_type}

        except Exception as e:
            print(f"[WARN] {fname}: {e}")

    if not mats:
        return pd.DataFrame(), {}

    mat = pd.concat(mats, axis=1)

    # Collapse duplicate samples
    mat = mat.groupby(level=0, axis=1).mean()

    # Log transform
    mat = np.log2(mat + 1)

    return mat, sample_info


def load_mirna_matrix_cached(project: str):
    cache_file = OUT_DIR / f"{project}_mirna_matrix.pkl"

    if cache_file.exists():
        print(f"[cache] matrix: {project}")
        return pd.read_pickle(cache_file)

    print(f"[build] matrix: {project}")
    mat, sample_info = load_mirna_matrix(project)

    pd.to_pickle((mat, sample_info), cache_file)

    return mat, sample_info


# =========================
# Statistics
# =========================

def fdr_bh(pvals):
    """
    Benjamini-Hochberg FDR correction.
    """
    pvals = np.asarray(pvals)
    n = len(pvals)

    order = np.argsort(pvals)
    ranked = pvals[order]

    fdr = ranked * n / (np.arange(1, n + 1))
    fdr = np.minimum.accumulate(fdr[::-1])[::-1]

    out = np.empty_like(fdr)
    out[order] = fdr

    return np.clip(out, 0, 1)


# =========================
# GDC File Retrieval
# =========================

def get_mirna_file_ids(project: str):
    cache_file = OUT_DIR / f"{project}_mirna_files.json"

    if cache_file.exists():
        print(f"[cache] miRNA file IDs: {project}")
        with open(cache_file) as f:
            return json.load(f)

    print(f"[download] miRNA file IDs: {project}")

    params = {
        "filters": {
            "op": "and",
            "content": [
                {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
                {"op": "in", "content": {"field": "data_type", "value": ["miRNA Expression Quantification"]}}
            ]
        },
        "fields": "file_id,file_name",
        "format": "JSON",
        "size": 200
    }

    r = requests.post(GDC_API_FILES, json=params)
    r.raise_for_status()

    hits = r.json()["data"]["hits"]
    file_ids = [(f["file_id"], f["file_name"]) for f in hits]

    with open(cache_file, "w") as f:
        json.dump(file_ids, f)

    return file_ids


# =========================
# Download Utilities
# =========================

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import gzip
import re

def download_one(file_id, file_name, project_dir):
    file_path = project_dir / file_name

    if file_path.exists() and file_path.stat().st_size > 0:
        return file_path   # ✅ return path

    url = f"{GDC_API_DATA}/{file_id}"

    try:
        r = requests.get(url, stream=True, timeout=60)
        if r.status_code != 200:
            return None

        with open(file_path, "wb") as f:
            for chunk in r.iter_content(8192):
                if chunk:
                    f.write(chunk)

        return file_path   # ✅ return path

    except Exception as e:
        print(f"[WARN] {file_name}: {e}")
        return None
    

def download_mirna_files(file_ids, project: str):
    project_dir = DATA_DIR / project
    project_dir.mkdir(parents=True, exist_ok=True)

    with ThreadPoolExecutor(MAX_WORKERS) as executor:
        futures = [
            executor.submit(download_one, fid, fname, project_dir)
            for fid, fname in file_ids
        ]

        for _ in tqdm(as_completed(futures),
                      total=len(futures),
                      desc=f"Downloading {project}"):
            pass


# =========================
# Alternative DE (t-test)
# =========================

from scipy.stats import ttest_ind


def process_mirna(mat: pd.DataFrame, min_var: float = 1e-5):
    print(f"[DEBUG] miRNA matrix: {mat.shape}")

    if mat.empty:
        return pd.DataFrame()

    # Variance filter
    mat = mat[mat.var(axis=1) > min_var]
    print(f"[DEBUG] After filtering: {mat.shape}")

    if mat.shape[1] < 6:
        print("[WARN] too few samples")
        return pd.DataFrame()

    # ⚠️ fallback split (ONLY if no labels exist)
    n = mat.shape[1]
    tumor = mat.iloc[:, : n // 2]
    normal = mat.iloc[:, n // 2 :]

    mu_T = tumor.mean(axis=1)
    mu_N = normal.mean(axis=1)

    logFC = mu_T - mu_N

    _, pvals = ttest_ind(
        tumor.T,
        normal.T,
        axis=0,
        equal_var=False,
        nan_policy="omit"
    )

    pval = pd.Series(pvals, index=mat.index).fillna(1.0)
    fdr = pd.Series(fdr_bh(pval.values), index=mat.index)

    score = logFC * (-np.log10(fdr + 1e-10))

    return pd.DataFrame({
        "mirna": mat.index,
        "logFC": logFC,
        "pval": pval,
        "fdr": fdr,
        "score": score
    }).sort_values("fdr")


# =========================
# File Metadata Mapping
# =========================

def build_file_id_map(file_ids):
    ids = [fid for fid, _ in file_ids]

    params = {
        "filters": {
            "op": "in",
            "content": {
                "field": "files.file_id",
                "value": ids
            }
        },
        "fields": "file_id,cases.samples.submitter_id,cases.samples.sample_type",
        "format": "JSON",
        "size": len(ids)
    }

    res = requests.post(GDC_API_FILES, json=params)
    res.raise_for_status()
    hits = res.json()["data"]["hits"]

    mapping = {}

    for hit in hits:
        fid = hit["file_id"]

        try:
            sample = hit["cases"][0]["samples"][0]
            barcode = sample["submitter_id"]
            stype = sample["sample_type"]

            if stype == "Primary Tumor":
                t = "tumor"
            elif stype == "Solid Tissue Normal":
                t = "normal"
            else:
                t = "other"

            mapping[fid] = {"barcode": barcode, "type": t}

        except (KeyError, IndexError, TypeError):
            mapping[fid] = {"barcode": None, "type": None}

    return mapping


# =========================
# GENCODE Handling
# =========================

def get_latest_gencode_url():
    base = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"

    r = requests.get(base)
    r.raise_for_status()

    versions = re.findall(r"release_(\d+)", r.text)
    latest = max(map(int, versions))

    url = f"{base}/release_{latest}/gencode.v{latest}.annotation.gtf.gz"

    print(f"[INFO] Latest GENCODE: v{latest}")
    return url, f"v{latest}"


GENCODE_URL, GENCODE_VERSION = get_latest_gencode_url()
GTF_PATH = OUT_DIR / f"gencode.{GENCODE_VERSION}.annotation.gtf.gz"



def download_gtf():
    if GTF_PATH.exists() and GTF_PATH.stat().st_size > 1e6:
        print("[cache] GTF")
        return GTF_PATH

    print("[download] GTF")

    r = requests.get(GENCODE_URL, stream=True)
    r.raise_for_status()

    with open(GTF_PATH, "wb") as f:
        for chunk in r.iter_content(8192):
            if chunk:
                f.write(chunk)

    return GTF_PATH


def parse_gtf():
    path = download_gtf()

    gene_map = {}
    gene_annot = {}

    with gzip.open(path, "rt") as f:
        for line in tqdm(f, desc="Parsing GTF"):
            if line.startswith("#"):
                continue

            parts = line.split("\t")
            if parts[2] != "gene":
                continue

            chrom, start, end = parts[0], int(parts[3]), int(parts[4])
            attr = parts[8]

            gid = re.search(r'gene_id "([^"]+)"', attr)
            gname = re.search(r'gene_name "([^"]+)"', attr)
            gtype = re.search(r'gene_type "([^"]+)"', attr)

            if not (gid and gname and gtype):
                continue

            gid = gid.group(1).split(".")[0]
            gname = gname.group(1)
            gtype = gtype.group(1)

            if gtype != "protein_coding":
                continue

            gene_map[gid] = gname
            gene_annot[gname] = (chrom, start, end)

    gene_list = sorted(gene_annot.keys())

    print(f"[INFO] {len(gene_list)} genes loaded")
    return gene_list, gene_map, gene_annot


# =========================
# Helpers
# =========================

def minmax(s: pd.Series):
    s = s.fillna(0)
    return (s - s.min()) / (s.max() - s.min() + 1e-12)


def process_mirna_cached(mat, sample_info, cancer: str):
    cache_file = OUTPUT_DIR / f"{cancer}_mirna_scores.csv"

    if cache_file.exists():
        print(f"[cache] scores: {cancer}")
        return pd.read_csv(cache_file)

    print(f"[compute] DE: {cancer}")

    df = process_mirna_limma(mat, sample_info)

    df.to_csv(cache_file, index=False)
    return df



def build_sample_labels(samples):
    labels = {}

    for s in samples:
        try:
            code = int(s.split("-")[3][:2])
            labels[s] = "tumor" if code < 10 else "normal"
        except:
            labels[s] = "unknown"

    return labels


def map_mirna_to_genes_weighted(mirna_series, gene_list, mirna_targets):
    gene_scores = {g: 0.0 for g in gene_list}
    gene_weights = {g: 0.0 for g in gene_list}

    for mir, val in mirna_series.items():
        mir = mir.lower().strip()

        if mir not in mirna_targets:
            continue

        for gene, weight in mirna_targets[mir]:
            if gene in gene_scores:
                gene_scores[gene] += val * weight
                gene_weights[gene] += weight

    return pd.Series({
        g: gene_scores[g] / gene_weights[g] if gene_weights[g] > 0 else 0.0
        for g in gene_list
    })





def get_latest_gencode_url():
    base = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"

    r = requests.get(base)
    r.raise_for_status()

    versions = re.findall(r"release_(\d+)", r.text)
    latest = max(map(int, versions))

    url = f"{base}/release_{latest}/gencode.v{latest}.annotation.gtf.gz"

    print(f"[INFO] Latest GENCODE: v{latest}")
    return url, f"v{latest}"


GENCODE_URL, GENCODE_VERSION = get_latest_gencode_url()
GTF_PATH = OUT_DIR / f"gencode.{GENCODE_VERSION}.annotation.gtf.gz"


# =========================================================
# UTILITIES
# =========================================================
def normalize_mirna(m):
    m = str(m).lower().strip()
    m = m.replace("hsa-", "")
    m = m.replace("mirna", "mir")
    m = m.replace("-5p", "").replace("-3p", "")

    parts = m.split("-")
    return "-".join(parts[:2]) if len(parts) >= 2 else m

# =========================
# File Metadata Mapping
# =========================

def build_file_id_map(file_ids):
    ids = [fid for fid, _ in file_ids]

    params = {
        "filters": {
            "op": "in",
            "content": {
                "field": "files.file_id",
                "value": ids
            }
        },
        "fields": "file_id,cases.samples.submitter_id,cases.samples.sample_type",
        "format": "JSON",
        "size": len(ids)
    }

    res = requests.post(GDC_API_FILES, json=params)
    res.raise_for_status()
    hits = res.json()["data"]["hits"]

    mapping = {}

    for hit in hits:
        fid = hit["file_id"]

        try:
            sample = hit["cases"][0]["samples"][0]
            barcode = sample["submitter_id"]
            stype = sample["sample_type"]

            if stype == "Primary Tumor":
                t = "tumor"
            elif stype == "Solid Tissue Normal":
                t = "normal"
            else:
                t = "other"

            mapping[fid] = {"barcode": barcode, "type": t}

        except (KeyError, IndexError, TypeError):
            mapping[fid] = {"barcode": None, "type": None}

    return mapping


def query_files(project, data_type, size=100):
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
            {"op": "in", "content": {"field": "data_type", "value": [data_type]}},
        ],
    }

    params = {
        "filters": json.dumps(filters),
        "format": "JSON",
        "size": str(size),
        "fields": "file_id,file_name,cases.samples.submitter_id",
    }

    r = requests.get(GDC_API_FILES, params=params)
    r.raise_for_status()

    hits = r.json()["data"]["hits"]

    out = []
    for h in hits:
        fid = h["file_id"]
        fname = h["file_name"]

        try:
            sample = h["cases"][0]["samples"][0]["submitter_id"]
        except:
            sample = None

        out.append((fid, fname, sample))

    return out

def download_one(fid, fname, out_dir):
    path = out_dir / fname

    if path.exists() and path.stat().st_size > 1000:
        return path

    url = f"{GDC_API_DATA}/{fid}"

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)

    return path


def download_gtf():
    if GTF_PATH.exists() and GTF_PATH.stat().st_size > 1e6:
        print("[cache] GTF")
        return GTF_PATH

    print("[download] GTF")

    r = requests.get(GENCODE_URL, stream=True)
    r.raise_for_status()

    with open(GTF_PATH, "wb") as f:
        for chunk in r.iter_content(8192):
            if chunk:
                f.write(chunk)

    return GTF_PATH


def parse_gtf():
    path = download_gtf()

    gene_map = {}
    gene_annot = {}

    with gzip.open(path, "rt") as f:
        for line in tqdm(f, desc="Parsing GTF"):
            if line.startswith("#"):
                continue

            parts = line.split("\t")
            if parts[2] != "gene":
                continue

            chrom, start, end = parts[0], int(parts[3]), int(parts[4])
            attr = parts[8]

            gid = re.search(r'gene_id "([^"]+)"', attr)
            gname = re.search(r'gene_name "([^"]+)"', attr)
            gtype = re.search(r'gene_type "([^"]+)"', attr)

            if not (gid and gname and gtype):
                continue

            gid = gid.group(1).split(".")[0]
            gname = gname.group(1)
            gtype = gtype.group(1)

            if gtype != "protein_coding":
                continue

            gene_map[gid] = gname
            gene_annot[gname] = (chrom, start, end)

    gene_list = sorted(gene_annot.keys())

    print(f"[INFO] {len(gene_list)} genes loaded")
    return gene_list, gene_map, gene_annot


def parallel_download(files, subdir):
    paths = []
    out_dir = DATA_DIR / subdir
    out_dir.mkdir(parents=True, exist_ok=True)

    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        futures = {
            ex.submit(download_one, fid, fname, out_dir): (fname, sample)
            for fid, fname, sample in files
        }

        for f in as_completed(futures):
            fname, sample = futures[f]
            try:
                path = f.result()
                paths.append((path, sample))
            except Exception as e:
                print("[ERROR]", fname, e)

    return paths


def minmax(s: pd.Series):
    s = s.fillna(0)
    return (s - s.min()) / (s.max() - s.min() + 1e-12)


def process_mirna_limma(mat: pd.DataFrame,
                       sample_info: dict,
                       min_var: float = 1e-5) -> pd.DataFrame:
    print(f"[DEBUG] matrix: {mat.shape}")

    if mat.empty:
        return pd.DataFrame()

    # Variance filter
    mat = mat[mat.var(axis=1) > min_var]
    print(f"[DEBUG] after variance filter: {mat.shape}")

    group = []
    keep_cols = []

    for sample in mat.columns:
        sample_type = sample_info.get(sample, {}).get("type", "other")
        if sample_type in ("tumor", "normal"):
            group.append(1 if sample_type == "tumor" else 0)
            keep_cols.append(sample)

    if len(keep_cols) < 6:
        print("[WARN] not enough tumor/normal samples")
        return pd.DataFrame()

    mat = mat[keep_cols]
    group = np.array(group)

    print(f"[INFO] tumor: {(group==1).sum()} | normal: {(group==0).sum()}")

    # Batch correction
    mat = combat_batch_correction(mat)

    # Design matrix
    X = sm.add_constant(group)

    logFC, pvals = [], []

    for gene in mat.index:
        y = mat.loc[gene].values

        try:
            model = sm.OLS(y, X).fit()
            logFC.append(model.params[1])
            pvals.append(model.pvalues[1])
        except Exception:
            logFC.append(0.0)
            pvals.append(1.0)

    logFC = pd.Series(logFC, index=mat.index)
    pval = pd.Series(pvals, index=mat.index)

    # FDR correction
    _, fdr_vals, _, _ = multipletests(pval.values, method="fdr_bh")
    fdr = pd.Series(fdr_vals, index=mat.index)

    score = logFC * (-np.log10(fdr + 1e-10))

    return pd.DataFrame({
        "mirna": mat.index,
        "logFC": logFC,
        "pval": pval,
        "fdr": fdr,
        "score": score
    }).sort_values("fdr")



def process_mirna_cached(mat, sample_info, cancer: str):
    cache_file = OUTPUT_DIR / f"{cancer}_mirna_scores.csv"

    if cache_file.exists():
        print(f"[cache] scores: {cancer}")
        return pd.read_csv(cache_file)

    print(f"[compute] DE: {cancer}")

    df = process_mirna_limma(mat, sample_info)

    df.to_csv(cache_file, index=False)
    return df

def _map_mirna_to_genes_weighted(mirna_series, gene_list, mirna_targets):
    gene_scores = {g: 0.0 for g in gene_list}
    gene_weights = {g: 0.0 for g in gene_list}

    for mir, val in mirna_series.items():
        mir = mir.lower().strip()

        if mir not in mirna_targets:
            continue

        for gene, weight in mirna_targets[mir]:
            if gene in gene_scores:
                gene_scores[gene] += val * weight
                gene_weights[gene] += weight

    return pd.Series({
        g: gene_scores[g] / gene_weights[g] if gene_weights[g] > 0 else 0.0
        for g in gene_list
    })

def _map_mirna_to_genes_weighted(mirna_series, gene_list, mirna_targets):
    gene_scores = {g: 0.0 for g in gene_list}
    gene_weights = {g: 0.0 for g in gene_list}

    matched = 0

    for mir, val in mirna_series.items():
        mir = canonical_mirna(mir)

        if mir not in mirna_targets:
            continue

        matched += 1

        for gene, weight in mirna_targets[mir]:
            if gene in gene_scores:
                gene_scores[gene] += val * weight
                gene_weights[gene] += weight

    print(f"[DEBUG] matched miRNAs (score): {matched}/{len(mirna_series)}")

    return pd.Series({
        g: gene_scores[g] / gene_weights[g] if gene_weights[g] > 0 else 0.0
        for g in gene_list
    })

def get_sample_info_from_gdc(file_id: str):
    """
    Returns (barcode, type)
    """
    meta = query_gdc(file_id) 

    try:
        sample = meta["cases"][0]["samples"][0]

        barcode = sample["submitter_id"]
        stype = sample["sample_type"]

        if stype == "Primary Tumor":
            return barcode, "tumor"
        elif stype == "Solid Tissue Normal":
            return barcode, "normal"
        return barcode, "other"

    except (KeyError, IndexError, TypeError):
        return None, None


def load_mirna_matrix_3(project: str,
                      file_ids: list,
                      file_map: dict):
    """
    Build miRNA expression matrix for a project.
    """
    cancer_dir = DATA_DIR / project

    mats = []
    sample_info = {}

    for fid, fname in file_ids:
        fpath = cancer_dir / fname

        if not fpath.exists():
            continue

        try:
            # df = pd.read_csv(fpath, sep="\t", low_memory=False)
            try:
                df = pd.read_csv(fpath, sep="\t", low_memory=False)
                if df.shape[1] == 1:
                    raise ValueError("Wrong delimiter")
            except:
                df = pd.read_csv(fpath, low_memory=False)

            # mirna_col = next((c for c in df.columns if "mirna" in c.lower()), None)
            # expr_col = next((c for c in df.columns if "read" in c.lower() or "rpm" in c.lower()), None)

            mirna_col = next((c for c in df.columns if "mir" in c.lower()), None)
            expr_col = next((c for c in df.columns if any(x in c.lower() for x in ["read", "rpm", "count"])), None)

            if not mirna_col or not expr_col:
                continue

            df = df[[mirna_col, expr_col]].dropna()

            # barcode, sample_type = get_sample_info_from_gdc(fid)
            info = file_map.get(fid, {})
            barcode = info.get("barcode")
            sample_type = info.get("type", "other")

            if barcode is None:
                barcode = f"unknown_{fid}"
                sample_type = "other"

            df.columns = ["miRNA", barcode]
            df = df.set_index("miRNA")

            mats.append(df)
            sample_info[barcode] = {"type": sample_type}

        except Exception as e:
            print(f"[WARN] {fname}: {e}")

    if not mats:
        return pd.DataFrame(), {}

    mat = pd.concat(mats, axis=1)

    # Collapse duplicate samples
    mat = mat.groupby(level=0, axis=1).mean()

    # Log transform
    mat = np.log2(mat + 1)

    return mat, sample_info

# =========================================================
# LOAD miRNA MATRIX (ROBUST)
# =========================================================
def load_mirna_matrix(paths):

    mats = []

    for f, sample_id in paths:

        if sample_id is None:
            continue

        try:
            df = pd.read_csv(f, sep="\t", comment="#", low_memory=False)

            if "miRNA_ID" not in df.columns:
                continue

            if "reads_per_million_miRNA_mapped" in df.columns:
                val_col = "reads_per_million_miRNA_mapped"
            elif "read_count" in df.columns:
                val_col = "read_count"
            else:
                continue

            tmp = df[["miRNA_ID", val_col]].copy()
            tmp.columns = ["mirna", "value"]
            tmp["sample"] = sample_id  # ✅ FIXED

            mats.append(tmp)

        except Exception as e:
            print(f"[ERROR reading] {f.name}: {e}")

    if not mats:
        return pd.DataFrame()

    mat = pd.concat(mats, ignore_index=True)

    # mat["mirna"] = mat["mirna"].apply(normalize_mirna)
    mat["mirna"] = mat["mirna"].apply(canonical_mirna)

    mat = mat.pivot_table(
        index="mirna",
        columns="sample",
        values="value",
        aggfunc="mean"
    )

    print(f"[INFO] Loaded matrix: {mat.shape}")

    return np.log2(mat + 1)

def load_mirtarbase(mirtarbase_files):
    """
    Load miRTarBase files and return:
        mirna -> list of (gene, weight)
    """
    mapping = {}

    if isinstance(mirtarbase_files, str):
        mirtarbase_files = {"default": (mirtarbase_files, 1.0)}

    for name, (path, weight) in mirtarbase_files.items():
        try:
            df = pd.read_excel(path) if path.endswith(".xlsx") else pd.read_csv(path)

            mir_col = next((c for c in df.columns if "mirna" in c.lower()), None)
            gene_col = next((c for c in df.columns if "target" in c.lower()), None)

            if not mir_col or not gene_col:
                print(f"[WARN] Skipping {name}: missing columns")
                continue

            for _, row in df.iterrows():
                mir = str(row[mir_col]).lower().strip()
                gene = str(row[gene_col]).upper().strip()

                if mir and gene and gene != "NAN":
                    mapping.setdefault(mir, []).append((gene, weight))

            print(f"[INFO] Loaded {name}: {len(df)} rows")

        except Exception as e:
            print(f"[WARN] Failed {name}: {e}")

    print(f"[INFO] Total miRNAs loaded: {len(mapping)}")
    return mapping

def get_mirna_file_ids(project: str):
    cache_file = OUT_DIR / f"{project}_mirna_files.json"

    if cache_file.exists():
        print(f"[cache] miRNA file IDs: {project}")
        with open(cache_file) as f:
            return json.load(f)

    print(f"[download] miRNA file IDs: {project}")

    params = {
        "filters": {
            "op": "and",
            "content": [
                {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
                {"op": "in", "content": {"field": "data_type", "value": ["miRNA Expression Quantification"]}}
            ]
        },
        "fields": "file_id,file_name",
        "format": "JSON",
        "size": 200
    }

    r = requests.post(GDC_API_FILES, json=params)
    r.raise_for_status()

    hits = r.json()["data"]["hits"]
    file_ids = [(f["file_id"], f["file_name"]) for f in hits]

    with open(cache_file, "w") as f:
        json.dump(file_ids, f)

    return file_ids

def download_mirna_files(file_ids, project: str):
    project_dir = DATA_DIR / project
    project_dir.mkdir(parents=True, exist_ok=True)

    with ThreadPoolExecutor(MAX_WORKERS) as executor:
        futures = [
            executor.submit(download_one, fid, fname, project_dir)
            for fid, fname in file_ids
        ]

        for _ in tqdm(as_completed(futures),
                      total=len(futures),
                      desc=f"Downloading {project}"):
            pass




def _map_mirna_to_gene_pvalues(mirna_pvals, gene_list, mirna_targets):
    gene_pvals = {}

    for gene in gene_list:
        gene_pvals[gene] = []

    # Collect p-values per gene

    mapped_edges = 0

    for mir, pval in mirna_pvals.items():
        mir = mir.lower().strip()
        if mir not in mirna_targets:
            continue

        for gene, _ in mirna_targets[mir]:
            if gene in gene_list:
                mapped_edges += 1
                gene_pvals[gene].append(pval)

    print(f"[DEBUG] total mapped miRNA→gene edges: {mapped_edges}")


    combined = {}

    for gene, pvals in gene_pvals.items():
        if len(pvals) == 0:
            combined[gene] = 1.0
        else:
            try:
                stat, p = combine_pvalues(pvals, method="fisher")
                combined[gene] = p
            except:
                combined[gene] = 1.0

    return pd.Series(combined)


def map_mirna_to_gene_pvalues_(mirna_pvals, gene_list, mirna_targets):
    gene_pvals = {g: [] for g in gene_list}

    matched = 0

    for mir, pval in mirna_pvals.items():
        mir = canonical_mirna(mir)

        if mir not in mirna_targets:
            continue

        matched += 1

        for gene, _ in mirna_targets[mir]:
            if gene in gene_pvals:
                gene_pvals[gene].append(pval)

    print(f"[DEBUG] matched miRNAs (pval): {matched}/{len(mirna_pvals)}")

    combined = {}

    for gene, pvals in gene_pvals.items():
        if len(pvals) == 0:
            combined[gene] = 1.0
        else:
            try:
                _, p = combine_pvalues(pvals, method="fisher")
                combined[gene] = p
            except:
                combined[gene] = 1.0

    return pd.Series(combined)

from scipy.stats import combine_pvalues
import numpy as np

def map_mirna_to_gene_pvalues(mirna_pvals, gene_list, mirna_targets):

    gene_pvals = {g: [] for g in gene_list}
    matched = 0

    for mir, pval in mirna_pvals.items():
        mir = canonical_mirna(mir)

        if mir not in mirna_targets:
            continue

        matched += 1

        for gene, _ in mirna_targets[mir]:
            if gene in gene_pvals:
                gene_pvals[gene].append(pval)

    print(f"[DEBUG] matched miRNAs (pval): {matched}/{len(mirna_pvals)}")

    combined = {}

    for gene, pvals in gene_pvals.items():

        # 🔥 CLEAN p-values
        pvals = [
            p for p in pvals
            if (p is not None) and (not np.isnan(p)) and (p < 0.999)
        ]

        # 🔥 require at least 1–2 signals
        if len(pvals) == 0:
            combined[gene] = 1.0
            continue

        # Optional: require stronger evidence
        # if len(pvals) < 2:
        #     combined[gene] = 1.0
        #     continue

        try:
            _, p = combine_pvalues(pvals, method="fisher")
            combined[gene] = min(max(p, 1e-300), 1.0)  # clamp
        except:
            combined[gene] = 1.0

    return pd.Series(combined)
# =========================================================
# SAMPLE LABELS
# =========================================================
def build_sample_labels(samples):
    labels = {}

    for s in samples:
        try:
            code = int(s.split("-")[3][:2])
            labels[s] = "tumor" if code < 10 else "normal"
        except:
            labels[s] = "unknown"

    return labels



# =========================================================
# PROCESS miRNA (STABLE)
# =========================================================
def process_mirna_fast(mat):

    if mat.empty:
        return pd.DataFrame()

    labels = build_sample_labels(mat.columns)

    tumor = [s for s in mat.columns if labels[s] == "tumor"]
    normal = [s for s in mat.columns if labels[s] == "normal"]

    print(f"[DEBUG] tumor={len(tumor)} normal={len(normal)}")

    # -------------------------
    # FALLBACK (important)
    # -------------------------
    if len(tumor) < 3 or len(normal) < 2:
        print("[WARN] fallback: using mean expression")

        score = mat.mean(axis=1)

        return pd.DataFrame({
            "score": score,
            "fdr": pd.Series(1, index=score.index)
        })

    # -------------------------
    # NORMAL CASE
    # -------------------------
    X = mat[tumor]
    Y = mat[normal]

    _, pvals = ttest_ind(X.values, Y.values, axis=1,
                         equal_var=False, nan_policy="omit")

    pvals = pd.Series(pvals, index=mat.index).fillna(1.0)

    _, fdr_vals, _, _ = multipletests(pvals.values, method="fdr_bh")
    fdr = pd.Series(fdr_vals, index=pvals.index)

    logFC = X.mean(axis=1) - Y.mean(axis=1)

    weights = -np.log10(np.maximum(pvals, 1e-300))

    score = logFC * weights

    return pd.DataFrame({
        "score": score,
        "logFC": logFC,
        "pval": pvals,
        "fdr": fdr
    })


# =========================================================
# miRNA NAMING (ROBUST + CONSISTENT)
# =========================================================

def canonical_mirna(m):
    """
    Standard canonical form:
    - lowercase
    - remove 'hsa-'
    - unify mir/mirna
    - normalize separators
    - KEEP 5p/3p (important biologically)
    """
    m = str(m).lower().strip()

    # remove species prefix
    if m.startswith("hsa-"):
        m = m[4:]

    # unify naming
    m = m.replace("mirna", "mir")

    # normalize separators
    m = m.replace("_", "-")

    return m


def generate_mirna_aliases(m):
    """
    Generate all equivalent representations of a miRNA
    """
    m = canonical_mirna(m)

    aliases = set()
    aliases.add(m)

    # remove arm (mir-21-5p → mir-21)
    if m.endswith("-5p") or m.endswith("-3p"):
        base = m.rsplit("-", 1)[0]
        aliases.add(base)

    # add back hsa- versions
    aliases |= {"hsa-" + a for a in aliases}

    return aliases


# =========================================================
# MAIN PIPELINE
# =========================================================
mirna_all = {}

for cancer in CANCERS:
    print(f"\n=== {cancer} ===")

    files = query_files(cancer, DATA_TYPES["mirna"])
    paths = parallel_download(files, f"{cancer}/mirna")

    mat = load_mirna_matrix(paths)

    print(f"[INFO] downloaded {len(paths)} files")

    # mat = load_mirna_matrix(cancer)

    if mat.empty:
        print("[SKIP] empty matrix")
        continue

    df = process_mirna_fast(mat)

    if df.empty:
        print("[SKIP] no valid DE results")
        continue

    mirna_all[cancer] = df["score"]


# =========================================================
# MERGE
# =========================================================
miRNA_all = pd.DataFrame(mirna_all)

mask = miRNA_all.abs().sum(axis=1) > 0
miRNA_all = miRNA_all.loc[mask]

print(f"\n[FINAL] miRNAs kept:", len(miRNA_all))

# =========================================================
# SAVE
# =========================================================
miRNA_all.to_csv(OUTPUT_DIR / "mirna_features.csv")

print("✅ DONE")

# =========================================================
# 0. INIT
# =========================================================
print("[INIT] Loading gene annotations + miRNA targets...")

gene_list, gene_map, gene_annot = parse_gtf()
mirna_targets_raw = load_mirtarbase(MIRTARBASE_FILES)

def normalize_mirtarbase(mirna_targets_raw):
    normalized = {}

    for mir, targets in mirna_targets_raw.items():

        aliases = generate_mirna_aliases(mir)

        for alias in aliases:
            if alias not in normalized:
                normalized[alias] = []

            normalized[alias].extend(targets)

    return normalized


mirna_targets_raw = load_mirtarbase(MIRTARBASE_FILES)
mirna_targets = normalize_mirtarbase(mirna_targets_raw)

print(f"[INFO] miRTarBase normalized keys: {len(mirna_targets)}")

mirna_dict = {}  # cancer -> DE results


# =========================================================
# 1. MAIN PIPELINE (UNIFIED)
# =========================================================
for cancer in CANCERS:

    print(f"\n=== {cancer} ===")

    # -----------------------------
    # Step 1: Query + Download
    # -----------------------------
    files = query_files(cancer, DATA_TYPES["mirna"], size=200)
    print(f"[INFO] files found: {len(files)}")

    if not files:
        print("[SKIP] no files returned")
        continue

    paths = parallel_download(files, f"{cancer}/mirna")
    print(f"[INFO] downloaded: {len(paths)}")

    # -----------------------------
    # Step 2: Load Matrix
    # -----------------------------
    mat = load_mirna_matrix(paths)

    if mat.empty:
        print("[SKIP] empty matrix after loading")
        continue

    print(f"[INFO] matrix shape: {mat.shape}")

    # -----------------------------
    # Step 3: Differential Expression
    # -----------------------------
    df = process_mirna_fast(mat)

    if df.empty:
        print("[SKIP] empty DE result")
        continue

    print(f"[INFO] DE result shape: {df.shape}")

    # Ensure consistent format
    df["mirna"] = df.index
    df = df.reset_index(drop=True)

    # Sanity check
    if df["score"].abs().sum() == 0:
        print("[WARN] all-zero scores, skipping")
        continue

    mirna_dict[cancer] = df.copy()


# =========================================================
# 2. VALIDATION
# =========================================================
if not mirna_dict:
    raise RuntimeError("No valid miRNA results produced")

print(f"\n[INFO] cancers processed: {len(mirna_dict)}")


# =========================================================
# 3. BUILD GLOBAL MATRICES
# =========================================================
mirna_all_dict = {}
mirna_pval_dict = {}
mirna_fdr_dict = {}

for cancer, df in mirna_dict.items():
    df = df.set_index("mirna")

    mirna_all_dict[cancer] = df["score"]

    mirna_pval_dict[cancer] = df.get(
        "pval",
        pd.Series(1, index=df.index)
    )

    mirna_fdr_dict[cancer] = df.get(
        "fdr",
        pd.Series(1, index=df.index)
    )

mirna_all = pd.DataFrame(mirna_all_dict)
mirna_pval_all = pd.DataFrame(mirna_pval_dict)
print(mirna_pval_all.describe())
mirna_fdr_all = pd.DataFrame(mirna_fdr_dict)

print(f"[INFO] combined matrix: {mirna_all.shape}")


# =========================================================
# 4. FILTER SIGNIFICANT miRNAs
# =========================================================
mask = (mirna_fdr_all < 0.05).any(axis=1)

print(f"[INFO] significant miRNAs: {mask.sum()}")

# fallback if nothing passes
if mask.sum() == 0:
    print("[WARN] no miRNAs pass FDR → keeping top by variance")
    mask = mirna_all.var(axis=1) > 0


mirna_all = mirna_all.loc[mask]
mirna_pval_all = mirna_pval_all.loc[mask]
mirna_fdr_all = mirna_fdr_all.loc[mask]

print(f"[FINAL] miRNA matrix: {mirna_all.shape}")


# =========================================================
# 5. MAP miRNA → GENE
# =========================================================
print("\n[STEP] Mapping miRNA → gene...")

gene_mirna_dict = {}

for cancer in mirna_all.columns:

    mir_series = mirna_all[cancer].dropna()

    gene_scores = map_mirna_to_genes_weighted(
        mir_series,
        gene_list,
        mirna_targets
    )

    gene_mirna_dict[cancer] = gene_scores

gene_mirna_all = pd.DataFrame(gene_mirna_dict)

print(f"[INFO] gene matrix: {gene_mirna_all.shape}")


# =========================================================
# 6. SORT (REPRODUCIBILITY)
# =========================================================
def sort_df(df):
    return df.sort_index().sort_index(axis=1)

mirna_all = sort_df(mirna_all)
mirna_fdr_all = sort_df(mirna_fdr_all)
gene_mirna_all = sort_df(gene_mirna_all)


print("\n[STEP] Mapping miRNA → gene p-values...")

gene_pval_dict = {}

for cancer in mirna_all.columns:

    # miRNA p-values for this cancer
    mir_pvals = mirna_pval_all[cancer].dropna()

    gene_pvals = map_mirna_to_gene_pvalues(
        mir_pvals,
        gene_list,
        mirna_targets
    )

    gene_pval_dict[cancer] = gene_pvals

gene_mirna_pval_all = pd.DataFrame(gene_pval_dict)

print(f"[INFO] gene pval matrix: {gene_mirna_pval_all.shape}")

gene_mirna_all = gene_mirna_all.reindex(gene_list)
gene_mirna_pval_all = gene_mirna_pval_all.reindex(gene_list)

# =========================================================
# REMOVE ZERO / NON-INFORMATIVE ROWS
# =========================================================

# --- Features: remove genes with all zeros ---
mask_gene = (gene_mirna_all != 0).any(axis=1)
gene_mirna_all = gene_mirna_all.loc[mask_gene]

print(f"[FILTER] genes kept (features): {mask_gene.sum()}")

# --- P-values: remove genes with all pval = 1 ---
mask_pval = (gene_mirna_pval_all < 1).any(axis=1)
gene_mirna_pval_all = gene_mirna_pval_all.loc[mask_pval]

print(f"[FILTER] genes kept (pvals): {mask_pval.sum()}")


# =========================================================
# COMPUTE GENE-LEVEL FDR (from miRNA → gene)
# =========================================================
gene_mirna_fdr_all = pd.DataFrame(index=gene_mirna_pval_all.index)

for cancer in gene_mirna_pval_all.columns:
    pvals = gene_mirna_pval_all[cancer].fillna(1.0).values
    fdr = fdr_bh(pvals)
    gene_mirna_fdr_all[cancer] = fdr

print(f"[INFO] gene FDR matrix: {gene_mirna_fdr_all.shape}")


# # keep genes significant in at least one cancer
# mask_sig = (gene_mirna_pval_all < 0.05).any(axis=1)
mask_sig = (gene_mirna_fdr_all < 0.05).any(axis=1)

gene_mirna_all = gene_mirna_all.loc[mask_sig]
gene_mirna_pval_all = gene_mirna_pval_all.loc[mask_sig]

print(f"[FILTER] significant genes: {mask_sig.sum()}")


# =========================================================
# 7. SAVE OUTPUTS
# =========================================================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# files = {
#     "mirna_features.csv": mirna_all,
#     "mirna_pvalues.csv": mirna_pval_all,
#     "mirna_fdr.csv": mirna_fdr_all,
#     "gene_from_mirna_features.csv": gene_mirna_all,
#     "gene_from_mirna_pvalues.csv": gene_mirna_pval_all,
# }

files = {
    "mirna_features.csv": mirna_all,
    "mirna_pvalues.csv": mirna_pval_all,
    "mirna_fdr.csv": mirna_fdr_all,
    "gene_from_mirna_features.csv": gene_mirna_all,
    "gene_from_mirna_pvalues.csv": gene_mirna_pval_all,
    "gene_from_mirna_fdr.csv": gene_mirna_fdr_all,   # ✅ NEW
}

for fname, df in files.items():
    path = OUTPUT_DIR / fname

    if df.empty:
        print(f"[WARN] Skipping empty file: {fname}")
        continue

    df.to_csv(path)
    print(f"[SAVED] {path}")

print("✅ All outputs saved successfully")


c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Latest GENCODE: v49
[INFO] Latest GENCODE: v49

=== TCGA-BLCA ===
[INFO] Loaded matrix: (1881, 98)
[INFO] downloaded 100 files
[DEBUG] tumor=90 normal=8

=== TCGA-BRCA ===
[INFO] Loaded matrix: (1881, 100)
[INFO] downloaded 100 files
[DEBUG] tumor=94 normal=6

=== TCGA-CESC ===
[INFO] Loaded matrix: (1881, 100)
[INFO] downloaded 100 files
[DEBUG] tumor=99 normal=1
[WARN] fallback: using mean expression

=== TCGA-COAD ===
[INFO] Loaded matrix: (1881, 100)
[INFO] downloaded 100 files
[DEBUG] tumor=99 normal=1
[WARN] fallback: using mean expression

=== TCGA-ESCA ===
[INFO] Loaded matrix: (1881, 99)
[INFO] downloaded 100 files
[DEBUG] tumor=95 normal=4

=== TCGA-GBM ===
[INFO] Loaded matrix: (1881, 99)
[INFO] downloaded 100 files
[DEBUG] tumor=97 normal=2

=== TCGA-HNSC ===
[INFO] Loaded matrix: (1881, 100)
[INFO] downloaded 100 files
[DEBUG] tumor=92 normal=8

=== TCGA-KIRC ===
[INFO] Loaded matrix: (1881, 97)
[INFO] downloaded 100 files
[DEBUG] tumor=84 normal=13

=== TCGA-KIRP =

Parsing GTF: 7750159it [00:08, 966529.72it/s] 


[INFO] 20070 genes loaded
[INFO] Loaded SE_WR: 27595 rows
[INFO] Loaded SE_W: 18292 rows
[INFO] Loaded SE_R: 24530 rows
[INFO] Loaded WE_CLIP: 4613908 rows
[INFO] Loaded WE_OTHER: 30455 rows
[INFO] Total miRNAs loaded: 4842
[INFO] Loaded SE_WR: 27595 rows
[INFO] Loaded SE_W: 18292 rows
[INFO] Loaded SE_R: 24530 rows
[INFO] Loaded WE_CLIP: 4613908 rows
[INFO] Loaded WE_OTHER: 30455 rows
[INFO] Total miRNAs loaded: 4842
[INFO] miRTarBase normalized keys: 12088

=== TCGA-BLCA ===
[INFO] files found: 200
[INFO] downloaded: 200
[INFO] Loaded matrix: (1881, 198)
[INFO] matrix shape: (1881, 198)
[DEBUG] tumor=187 normal=11
[INFO] DE result shape: (1881, 4)

=== TCGA-BRCA ===
[INFO] files found: 200
[INFO] downloaded: 200
[INFO] Loaded matrix: (1881, 200)
[INFO] matrix shape: (1881, 200)
[DEBUG] tumor=184 normal=16
[INFO] DE result shape: (1881, 4)

=== TCGA-CESC ===
[INFO] files found: 200
[INFO] downloaded: 200
[INFO] Loaded matrix: (1881, 200)
[INFO] matrix shape: (1881, 200)
[DEBUG] tumor=